In [ ]:
# ============================================================
# INSTALL REQUIRED LIBRARIES
# ============================================================
!pip install indic-nlp-library transformers torch scikit-learn huggingface_hub stanza

import json
import re
import torch
import numpy as np
import stanza
from collections import defaultdict
from tqdm import tqdm
from indicnlp.tokenize import sentence_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from google.colab import files

In [ ]:
# ============================================================
# STEP 1 — Upload ambiguous words file (.txt)
# ============================================================
# Your .txt file with 22 words separated by commas
# Example: डोळा,मार्ग,पान,...
# ============================================================
print("=" * 60)
print("STEP 1: Upload AMBIGUOUS WORDS file (.txt)")
print("=" * 60)

uploaded = files.upload()
file_name = list(uploaded.keys())[0]
with open(file_name, "r", encoding="utf-8") as f:
    text = f.read()

ambiguous_words = [w.strip() for w in text.split(",") if w.strip()]
print(f"Total ambiguous words loaded: {len(ambiguous_words)}")
print("Words:", ambiguous_words)

STEP 1: Upload AMBIGUOUS WORDS file (.txt)


Saving WSD marathi maam words.txt to WSD marathi maam words.txt
Total ambiguous words loaded: 20
Words: ['डोळा', 'मार्ग', 'वाट', 'वाटणे', 'देणे', 'भाव', 'योग', 'अर्थ', 'नाव', 'वर', 'बरोबर', 'हलका', 'मंद', 'फोडणे', 'उडणे', 'वार', 'गंध', 'पाठ', 'अंक', 'गुण']


In [ ]:

# ============================================================
# STEP 2 — Upload seed sentences JSON file
# ============================================================
# Your LLM-generated seed file
# Gloss is extracted from here — no separate gloss file needed
#
# Expected format:
# {
#   "डोळा": [
#     {
#       "sense_id": 101,
#       "pos": "noun",
#       "gloss": "...",
#       "sentences": ["sent1", "sent2", ...]
#     }
#   ]
# }
# ============================================================
print("\n" + "=" * 60)
print("STEP 2: Upload SEED SENTENCES JSON file")
print("=" * 60)

uploaded_seed = files.upload()
seed_file_name = list(uploaded_seed.keys())[0]

with open(seed_file_name, "r", encoding="utf-8") as f:
    seed_data = json.load(f)

wordnet_senses = {}
gloss_lookup = {}

for word, senses in seed_data.items():
    if word in ambiguous_words:

        # Build gloss lookup directly from seed file
        gloss_lookup[word] = {}
        for sense in senses:
            sense_id = str(sense["sense_id"])
            gloss_lookup[word][sense_id] = sense.get("gloss", "")

        # Build senses list
        wordnet_senses[word] = [
            {
                "sense_id": str(sense["sense_id"]),
                "pos": sense.get("pos", "noun"),
                "sentences": sense["sentences"]
            }
            for sense in senses
        ]

print(f"Words loaded from seed file: {len(wordnet_senses)}")
for word, senses in wordnet_senses.items():
    print(f"  {word}: {len(senses)} senses")

# Quick gloss check
first_word = list(gloss_lookup.keys())[0]
print(f"\nSample gloss check for '{first_word}':")
for sid, gloss in gloss_lookup[first_word].items():
    print(f"  Sense {sid}: {gloss[:60]}...")


STEP 2: Upload SEED SENTENCES JSON file


Saving Seed_Sentence_gemeni.json to Seed_Sentence_gemeni.json
Words loaded from seed file: 20
  डोळा: 5 senses
  मार्ग: 3 senses
  वाट: 3 senses
  वाटणे: 3 senses
  देणे: 3 senses
  भाव: 3 senses
  योग: 3 senses
  अर्थ: 3 senses
  नाव: 3 senses
  वर: 3 senses
  बरोबर: 3 senses
  हलका: 3 senses
  मंद: 3 senses
  फोडणे: 3 senses
  उडणे: 3 senses
  वार: 3 senses
  गंध: 3 senses
  पाठ: 3 senses
  अंक: 3 senses
  गुण: 3 senses

Sample gloss check for 'डोळा':
  Sense 101: आपल्या शरीराचा तो अवयव ज्याच्या मदतीने आपण बघतो (Eye / Physi...
  Sense 102: बटाटा, ऊस किंवा इतर झाडांवर येणारा छोटा कोंब, ज्यातून नवीन र...
  Sense 103: अचानक झोप येणे किंवा थोडी डुलकी लागणे (Sleep / Nap)...
  Sense 104: एखाद्या व्यक्तीवर किंवा गोष्टीवर बारीक लक्ष ठेवणे, पाळत ठेवण...
  Sense 105: धान्य दळण्याच्या जुन्या जात्याला मध्यभागी असलेले भोक, ज्यातू...


In [ ]:

# ============================================================
# STEP 3 — Upload raw Marathi sentence files
# ============================================================
# Upload all 8-9 scraped news/article files at once
# ============================================================
print("\n" + "=" * 60)
print("STEP 3: Upload RAW MARATHI TEXT files (select all files)")
print("=" * 60)

uploaded_raw = files.upload()
raw_texts = []
for fname, content in uploaded_raw.items():
    raw_text = content.decode("utf-8")
    raw_texts.append(raw_text)
    print(f"  Loaded: {fname} ({len(raw_text)} characters)")

print(f"\nTotal raw files loaded: {len(raw_texts)}")


STEP 3: Upload RAW MARATHI TEXT files (select all files)


Saving raw_marathi_sentences_13.txt to raw_marathi_sentences_13.txt
Saving raw_marathi_sentences_12.txt to raw_marathi_sentences_12.txt
Saving raw_marathi_sentences_11.txt to raw_marathi_sentences_11.txt
Saving raw_marathi_sentences_10.txt to raw_marathi_sentences_10.txt
Saving raw_marathi_sentences_9.txt to raw_marathi_sentences_9.txt
Saving raw_marathi_sentences_8.txt to raw_marathi_sentences_8.txt
Saving raw_marathi_sentences_7.txt to raw_marathi_sentences_7.txt
Saving raw_marathi_sentences_6.txt to raw_marathi_sentences_6.txt
Saving raw_marathi_sentences_5.txt to raw_marathi_sentences_5.txt
Saving raw_marathi_sentences_4.txt to raw_marathi_sentences_4.txt
Saving raw_marathi_sentences_3.txt to raw_marathi_sentences_3.txt
Saving raw_marathi_sentences_2.txt to raw_marathi_sentences_2.txt
Saving raw_marathi_sentences (1).txt to raw_marathi_sentences (1).txt
Saving raw_marathi_sentences.txt to raw_marathi_sentences.txt
  Loaded: raw_marathi_sentences_13.txt (981381 characters)
  Loaded:

In [ ]:
# ============================================================
# STEP 4 — Clean and split raw text (UPDATED: Digit filter removed)
# ============================================================
# 1. Remove URLs
# 2. Remove non-Marathi characters (English, symbols)
# 3. Split into individual sentences
# 4. Keep only 5 to 30 word sentences (Digits are safely KEPT)
# 5. Remove duplicates
# ============================================================
print("\n" + "=" * 60)
print("STEP 4: Cleaning and splitting into sentences")
print("=" * 60)

def clean_text(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\n+", " ", text)
    text = re.sub(r"[^\u0900-\u097F\s.,?!।\d]", " ", text) # \d ठेवून अंक सुरक्षित केले
    text = re.sub(r"\s+", " ", text)
    return text.strip()

all_sentences = []
for text in raw_texts:
    cleaned = clean_text(text)
    sents = sentence_tokenize.sentence_split(cleaned, lang="mr")
    for s in sents:
        s = s.strip()
        words = s.split()

        # ✅ UPDATED: Only filter by length. We KEEP digits for words like 'गुण'
        if 5 <= len(words) <= 30:
            all_sentences.append(s)

all_sentences = list(set(all_sentences))
print(f"Total unique clean sentences: {len(all_sentences)}")


STEP 4: Cleaning and splitting into sentences
Total unique clean sentences: 40531


In [ ]:
# ============================================================
# STEP 5 — Map words to sentences using Stanza Lemmatizer
# ============================================================
# Why Stanza instead of prefix matching?
# Marathi words change form heavily with suffixes
# Stanza reduces every form back to base word
# Example: डोळ्याला → डोळा (exact match)
#
# Prefix matching is kept as FALLBACK
# If Stanza fails for any sentence → prefix match runs
# This ensures no sentence is lost due to Stanza errors
# ============================================================
print("\n" + "=" * 60)
print("STEP 5: Mapping words to sentences (Stanza Lemmatizer)")
print("=" * 60)

print("Downloading Stanza Marathi Model...")
stanza.download('mr', processors='tokenize,lemma')

# Safe GPU check — won't crash if no GPU available
use_gpu = torch.cuda.is_available()
print(f"GPU available: {use_gpu}")
nlp = stanza.Pipeline('mr', processors='tokenize,lemma', use_gpu=use_gpu)
print("Stanza loaded successfully")

word_sentence_map = defaultdict(list)
lemma_fail_count = 0

for sentence in tqdm(all_sentences, desc="Lemmatizing and Mapping"):
    try:
        doc = nlp(sentence)
        lemmas = []
        for sent in doc.sentences:
            for word_token in sent.words:
                if word_token.lemma:
                    lemmas.append(word_token.lemma)

        # Also get tokens for prefix fallback within same loop
        tokens = re.findall(r'[\u0900-\u097F]+', sentence)

        for target_word in wordnet_senses:
            stanza_match = target_word in lemmas
            prefix_match = any(token.startswith(target_word) for token in tokens)

            # Accept if either method finds the word
            if stanza_match or prefix_match:
                word_sentence_map[target_word].append(sentence)

    except Exception as e:
        # Full fallback to prefix matching if Stanza crashes
        lemma_fail_count += 1
        tokens = re.findall(r'[\u0900-\u097F]+', sentence)
        for target_word in wordnet_senses:
            if any(token.startswith(target_word) for token in tokens):
                word_sentence_map[target_word].append(sentence)

print(f"\nStanza failures (used prefix fallback): {lemma_fail_count}")
print("\nWords mapped to sentences:")
for word, sents in word_sentence_map.items():
    print(f"  {word}: {len(sents)} sentences")

# Warn if any word has 0 sentences
print()
for word in wordnet_senses:
    if len(word_sentence_map[word]) == 0:
        print(f"  ⚠ WARNING: '{word}' has 0 sentences — check your raw data")


STEP 5: Mapping words to sentences (Stanza Lemmatizer)


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Downloading these customized packages for language: mr (Marathi)...
| Processor | Package       |
-----------------------------
| tokenize  | ufal          |
| mwt       | ufal          |
| lemma     | ufal_nocharlm |



INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/mr/tokenize/ufal.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/mr/mwt/ufal.pt


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/mr/lemma/ufal_nocharlm.pt
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.11.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


GPU available: True


INFO:stanza:Downloaded file to /root/.cache/stanza/1.11.0/resources/resources.json
INFO:stanza:Loading these models for language: mr (Marathi):
| Processor | Package       |
-----------------------------
| tokenize  | ufal          |
| mwt       | ufal          |
| lemma     | ufal_nocharlm |

INFO:stanza:Using device: cuda
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: lemma
INFO:stanza:Done loading processors!


Stanza loaded successfully


Lemmatizing and Mapping: 100%|██████████| 40531/40531 [34:03<00:00, 19.84it/s]


Stanza failures (used prefix fallback): 0

Words mapped to sentences:
  वर: 6005 sentences
  देणे: 3533 sentences
  योग: 401 sentences
  मार्ग: 482 sentences
  वाट: 686 sentences
  डोळा: 91 sentences
  भाव: 383 sentences
  मंद: 258 sentences
  पाठ: 415 sentences
  वाटणे: 131 sentences
  वार: 347 sentences
  बरोबर: 94 sentences
  नाव: 526 sentences
  गुण: 137 sentences
  अर्थ: 428 sentences
  अंक: 83 sentences
  फोडणे: 6 sentences
  हलका: 24 sentences
  गंध: 21 sentences
  उडणे: 1 sentences



In [ ]:

# ============================================================
# STEP 6 — Load MuRIL model
# ============================================================
# Why MuRIL instead of IndicBERT?
# MuRIL was trained specifically on Indian languages
# It handles Marathi morphology much better
# IndicBERT breaks Marathi words into too many fragments
# ============================================================
print("\n" + "=" * 60)
print("STEP 6: HuggingFace login and loading MuRIL")
print("=" * 60)

from huggingface_hub import login
login()

from transformers import AutoTokenizer, AutoModel

model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
print("MuRIL loaded successfully")



STEP 6: HuggingFace login and loading MuRIL


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MuRIL loaded successfully


In [ ]:

# ============================================================
# STEP 7 — Token-level embedding function
# ============================================================
# Why token-level instead of sentence mean?
#
# Sentence mean (OLD — WRONG for WSD):
#   Averages ALL words → mixes kitchen + mother + eye together
#   Result is noisy and gives wrong sense matches
#
# Token-level (NEW — CORRECT for WSD):
#   MuRIL reads the whole sentence using Self-Attention first
#   Then each word's vector contains full sentence context
#   We extract ONLY the target word's vector
#   Result = "meaning of डोळा in THIS specific sentence"
# ============================================================
def embed(sentence, target_word):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    hidden_states = outputs.last_hidden_state[0]  # shape: [seq_len, 768]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Find which tokens belong to the target word
    target_indices = []
    for i, tok in enumerate(tokens):
        clean_tok = tok.replace("▁", "").replace("##", "")
        if target_word[:3] in clean_tok or clean_tok in target_word:
            target_indices.append(i)

    if target_indices:
        # Average only the target word's token vectors
        return hidden_states[target_indices].mean(dim=0, keepdim=True).numpy()
    else:
        # Fallback to sentence mean if word not found in tokens
        print(f"    ⚠ '{target_word}' not found in tokens — using sentence mean fallback")
        return hidden_states.mean(dim=0, keepdim=True).numpy()

In [ ]:
# ============================================================
# STEP 8 — Pre-compute seed sentence anchor vectors
# ============================================================
# For each sense:
# 1. Embed all 15 seed sentences using token-level extraction
# 2. Average them into ONE anchor vector
# This anchor vector = "typical meaning" of that sense
# Used in Step 9 to compare against raw sentences
# ============================================================
print("\n" + "=" * 60)
print("STEP 8: Computing Anchor Vectors from Seeds")
print("=" * 60)

seed_embeddings = {}

for word, senses in tqdm(wordnet_senses.items(), desc="Embedding seeds"):
    seed_embeddings[word] = {}
    for sense in senses:
        sense_id = sense["sense_id"]
        sense_vecs = [embed(sent, word) for sent in sense["sentences"]]
        seed_embeddings[word][sense_id] = np.mean(sense_vecs, axis=0)

print("Anchor vectors computed successfully")


STEP 8: Computing Anchor Vectors from Seeds


Embedding seeds:  20%|██        | 4/20 [00:17<01:05,  4.12s/it]

    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using

Embedding seeds:  25%|██▌       | 5/20 [00:21<00:59,  3.94s/it]

    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback


Embedding seeds:  90%|█████████ | 18/20 [01:10<00:07,  3.84s/it]

    ⚠ 'अंक' not found in tokens — using sentence mean fallback


Embedding seeds: 100%|██████████| 20/20 [01:17<00:00,  3.86s/it]

Anchor vectors computed successfully


In [ ]:

# ============================================================
# STEP 9 — Assign senses to raw sentences
# ============================================================
# For each raw sentence:
# 1. Extract target word vector using token-level embedding
# 2. Compare against each sense anchor vector
# 3. Pick sense with highest cosine similarity score
# 4. ONLY assign if score >= 0.65
#    If score is too low → discard the sentence
#    This prevents wrong labels from entering the dataset
# ============================================================
print("\n" + "=" * 60)
print("STEP 9: Assigning senses to raw sentences")
print("=" * 60)

SIMILARITY_THRESHOLD = 0.65
MIN_SENTENCES_PER_SENSE = 100

sense_sentence_map = defaultdict(lambda: defaultdict(list))
discarded_count = 0

for word, sentences in tqdm(word_sentence_map.items(), desc="Assigning senses"):
    if word not in wordnet_senses:
        continue

    senses = wordnet_senses[word]

    for sentence in sentences:
        sent_vec = embed(sentence, word)
        best_sense_id = None
        best_score = -1

        for sense in senses:
            sense_id = sense["sense_id"]
            seed_vec = seed_embeddings[word][sense_id]
            score = cosine_similarity(sent_vec, seed_vec)[0][0]
            if score > best_score:
                best_score = score
                best_sense_id = sense_id

        # Only assign if confidence is high enough
        if best_score >= SIMILARITY_THRESHOLD:
            sense_sentence_map[word][best_sense_id].append(sentence)
        else:
            discarded_count += 1

print(f"\nTotal discarded (score < {SIMILARITY_THRESHOLD}): {discarded_count}")
print("\nSentences assigned per sense:")
for word, senses in sense_sentence_map.items():
    for sense_id, sents in senses.items():
        print(f"  {word} sense {sense_id}: {len(sents)} sentences")


STEP 9: Assigning senses to raw sentences


Assigning senses:   0%|          | 0/20 [00:00<?, ?it/s]

    ⚠ 'वर' not found in tokens — using sentence mean fallback


Assigning senses:   5%|▌         | 1/20 [10:00<3:10:12, 600.66s/it]

    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using sentence mean fallback
    ⚠ 'देणे' not found in tokens — using

...

    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback
    ⚠ 'वार' not found in tokens — using sentence mean fallback


Assigning senses:  65%|██████▌   | 13/20 [21:39<04:06, 35.29s/it]

    ⚠ 'गुण' not found in tokens — using sentence mean fallback


Assigning senses: 100%|██████████| 20/20 [22:46<00:00, 68.30s/it]


Total discarded (score < 0.65): 0

Sentences assigned per sense:
  वर sense 1101: 5756 sentences
  वर sense 1102: 236 sentences
  वर sense 1103: 13 sentences
  देणे sense 503: 2484 sentences
  देणे sense 501: 661 sentences
  देणे sense 502: 388 sentences
  योग sense 701: 237 sentences
  योग sense 703: 155 sentences
  योग sense 702: 9 sentences
  मार्ग sense 201: 247 sentences
  मार्ग sense 203: 145 sentences
  मार्ग sense 202: 90 sentences
  वाट sense 303: 231 sentences
  वाट sense 301: 415 sentences
  वाट sense 302: 40 sentences
  डोळा sense 105: 49 sentences
  डोळा sense 103: 6 sentences
  डोळा sense 101: 36 sentences
  भाव sense 603: 121 sentences
  भाव sense 602: 192 sentences
  भाव sense 601: 70 sentences
  मंद sense 1403: 200 sentences
  मंद sense 1402: 46 sentences
  मंद sense 1401: 12 sentences
  पाठ sense 1901: 245 sentences
  पाठ sense 1903: 82 sentences
  पाठ sense 1902: 88 sentences
  वाटणे sense 402: 4 sentences
  वाटणे sense 401: 123 sentences
  वाटणे sense 403: 4 senten

In [ ]:
# ============================================================
# STEP 10 — Pad senses to reach 100 sentences minimum
# ============================================================
# If a sense has less than 100 sentences after Step 9:
# Re-score ALL sentences for that word
# Take top N closest ones above 0.60 threshold
# 0.60 is slightly relaxed from 0.65 — only for padding
# If still not enough → warn but never force bad sentences
# ============================================================
print("\n" + "=" * 60)
print("STEP 10: Padding senses to reach 100 sentences")
print("=" * 60)

PAD_THRESHOLD = 0.60

for word, senses in tqdm(wordnet_senses.items(), desc="Padding senses"):
    all_word_sentences = word_sentence_map.get(word, [])

    for sense in senses:
        sense_id = sense["sense_id"]
        current = sense_sentence_map[word][sense_id]
        deficit = MIN_SENTENCES_PER_SENSE - len(current)

        if deficit <= 0:
            print(f"  ✓ '{word}' sense {sense_id}: {len(current)} sentences")
            continue

        candidates = [s for s in all_word_sentences if s not in current]

        if not candidates:
            print(f"  ⚠ '{word}' sense {sense_id}: only {len(current)} — no candidates left")
            continue

        seed_vec = seed_embeddings[word][sense_id]
        scored = [
            (cosine_similarity(embed(s, word), seed_vec)[0][0], s)
            for s in candidates
        ]
        scored.sort(key=lambda x: x[0], reverse=True)

        # Only pad with sentences above PAD_THRESHOLD
        padded = [s for score, s in scored[:deficit] if score >= PAD_THRESHOLD]
        sense_sentence_map[word][sense_id].extend(padded)

        final_count = len(sense_sentence_map[word][sense_id])
        if final_count < MIN_SENTENCES_PER_SENSE:
            print(f"  ⚠ '{word}' sense {sense_id}: reached only {final_count} — need more raw data")
        else:
            print(f"  ✓ '{word}' sense {sense_id}: {final_count} sentences")



STEP 10: Padding senses to reach 100 sentences


Padding senses:   0%|          | 0/20 [00:00<?, ?it/s]

  ⚠ 'डोळा' sense 101: reached only 91 — need more raw data
  ⚠ 'डोळा' sense 102: reached only 91 — need more raw data
  ⚠ 'डोळा' sense 103: reached only 91 — need more raw data
  ⚠ 'डोळा' sense 104: reached only 91 — need more raw data


Padding senses:   5%|▌         | 1/20 [00:33<10:34, 33.41s/it]

  ⚠ 'डोळा' sense 105: reached only 91 — need more raw data
  ✓ 'मार्ग' sense 201: 247 sentences


...

  ✓ 'पाठ' sense 1903: 100 sentences
  ⚠ 'अंक' sense 2001: reached only 83 — need more raw data
  ⚠ 'अंक' sense 2002: reached only 83 — need more raw data


Padding senses:  95%|█████████▌| 19/20 [17:48<00:34, 34.63s/it]

  ⚠ 'अंक' sense 2003: reached only 83 — need more raw data
  ✓ 'गुण' sense 2101: 100 sentences
    ⚠ 'गुण' not found in tokens — using sentence mean fallback
  ✓ 'गुण' sense 2102: 100 sentences
    ⚠ 'गुण' not found in tokens — using sentence mean fallback


Padding senses: 100%|██████████| 20/20 [18:14<00:00, 54.74s/it]

  ✓ 'गुण' sense 2103: 100 sentences


In [ ]:
# ============================================================
# STEP 11 — Build final dataset rows (UPDATED: Injecting Golden Seeds)
# ============================================================
# Inject the 15 LLM-generated seed sentences per sense
# These are guaranteed perfect examples of each meaning
# Filter seeds with same rules as Step 4 before injecting
# ============================================================
print("\n" + "=" * 60)
print("STEP 11: Building final dataset & Injecting Golden Seeds")
print("=" * 60)

final_dataset = []

for word, senses in wordnet_senses.items():
    for sense in senses:
        sense_id = sense["sense_id"]

        # Inject golden seeds WITH filtering
        golden_sentences = sense["sentences"]
        injected_count = 0

        for g_sent in golden_sentences:
            g_sent = re.sub(r'\s+', ' ', g_sent).strip()
            g_words = g_sent.split()
            marathi_tokens = re.findall(r'[\u0900-\u097F]+', g_sent)

            # Word must be present in seed sentence
            word_present = any(
                token.startswith(word) for token in marathi_tokens
            )

            # ✅ UPDATED: Removed has_digits. Apply only Length and Word Presence.
            if (5 <= len(g_words) <= 30
                    and word_present
                    and g_sent not in sense_sentence_map[word][sense_id]):
                sense_sentence_map[word][sense_id].insert(0, g_sent)
                injected_count += 1

        print(f"  '{word}' sense {sense_id}: {injected_count} seeds injected")

        assigned_sentences = sense_sentence_map[word][sense_id]

        for sentence in assigned_sentences:
            for s in senses:
                label = 1 if s["sense_id"] == sense_id else 0
                gloss_text = gloss_lookup.get(word, {}).get(s["sense_id"], "")

                final_dataset.append({
                    "sentence": sentence,
                    "target_word": word,
                    "lemma": word,
                    "pos": s["pos"],
                    "sense_id": s["sense_id"],
                    "gloss": gloss_text,
                    "label": label
                })

print(f"\nRaw dataset size before saving: {len(final_dataset)}")


STEP 11: Building final dataset & Injecting Golden Seeds
  'डोळा' sense 101: 1 seeds injected
  'डोळा' sense 102: 7 seeds injected
  'डोळा' sense 103: 15 seeds injected
  'डोळा' sense 104: 15 seeds injected
  'डोळा' sense 105: 1 seeds injected
  'मार्ग' sense 201: 15 seeds injected
  'मार्ग' sense 202: 15 seeds injected
  'मार्ग' sense 203: 15 seeds injected
  'वाट' sense 301: 15 seeds injected
  'वाट' sense 302: 15 seeds injected
  'वाट' sense 303: 15 seeds injected
  'वाटणे' sense 401: 0 seeds injected
  'वाटणे' sense 402: 0 seeds injected
  'वाटणे' sense 403: 0 seeds injected
  'देणे' sense 501: 1 seeds injected
  'देणे' sense 502: 5 seeds injected
  'देणे' sense 503: 0 seeds injected
  'भाव' sense 601: 14 seeds injected
  'भाव' sense 602: 15 seeds injected
  'भाव' sense 603: 15 seeds injected
  'योग' sense 701: 15 seeds injected
  'योग' sense 702: 15 seeds injected
  'योग' sense 703: 14 seeds injected
  'अर्थ' sense 801: 15 seeds injected
  'अर्थ' sense 802: 15 seeds injected
  'अ

In [ ]:
# ============================================================
# STEP 12 — Clean and Save final dataset
# ============================================================
print("\n" + "=" * 60)
print("STEP 12: Finalizing and Saving dataset")
print("=" * 60)

# 12a — Normalize whitespace (Just to be safe before saving)
for item in final_dataset:
    item["sentence"] = re.sub(r'\s+', ' ', item["sentence"]).strip()

# NOTE: We DO NOT run prefix-matching checks here, because that would
# delete the irregular inflections that our Stanza lemmatizer successfully found!

# 12b — Save
output_path = "marathi_wsd_dataset_final.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, ensure_ascii=False, indent=2)

# Print final summary
print("\nDataset Summary (label=1 sentences per sense):")
summary = defaultdict(lambda: defaultdict(int))
for item in final_dataset:
    if item["label"] == 1:
        summary[item["target_word"]][item["sense_id"]] += 1

for word, senses in summary.items():
    print(f"\n  {word}:")
    for sense_id, count in senses.items():
        gloss_text = gloss_lookup.get(word, {}).get(sense_id, "")
        print(f"    Sense {sense_id} ({gloss_text[:50]}): {count} sentences")

print(f"\n✅ Dataset saved: {output_path}")
print(f"   Total entries: {len(final_dataset)}")
files.download(output_path)


STEP 12: Finalizing and Saving dataset

Dataset Summary (label=1 sentences per sense):

  डोळा:
    Sense 101 (आपल्या शरीराचा तो अवयव ज्याच्या मदतीने आपण बघतो (E): 92 sentences
    Sense 102 (बटाटा, ऊस किंवा इतर झाडांवर येणारा छोटा कोंब, ज्या): 98 sentences
    Sense 103 (अचानक झोप येणे किंवा थोडी डुलकी लागणे (Sleep / Nap): 106 sentences
    Sense 104 (एखाद्या व्यक्तीवर किंवा गोष्टीवर बारीक लक्ष ठेवणे,): 106 sentences
    Sense 105 (धान्य दळण्याच्या जुन्या जात्याला मध्यभागी असलेले भ): 92 sentences

  मार्ग:
    Sense 201 (एका ठिकाणाहून दुसऱ्या ठिकाणी जाण्यासाठी असलेला भौत): 262 sentences
    Sense 202 (एखादी समस्या सोडवण्याचा उपाय, किंवा कठीण परिस्थिती): 115 sentences
    Sense 203 (विचारप्रणाली, तत्त्वज्ञान, किंवा जीवन जगण्याची विश): 160 sentences

  वाट:
    Sense 301 (एका ठिकाणाहून दुसऱ्या ठिकाणी जाण्यासाठी असलेला रस्): 430 sentences
    Sense 302 (एखाद्या व्यक्तीची किंवा गोष्टीची प्रतीक्षा करणे (उ): 115 sentences
    Sense 303 (खूप मोठे नुकसान होणे, दुर्दशा होणे किंवा फसगत होणे): 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from collections import defaultdict
from google.colab import files

# 1. Prompt you to upload the dataset
print("=" * 50)
print("📤 Please upload 'marathi_wsd_dataset_final.json'")
print("=" * 50)

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2. Load the uploaded JSON file
with open(file_name, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# 3. Create a dictionary to store the counts
sentence_counts = defaultdict(lambda: defaultdict(int))

# 4. Count the sentences (Only counting the correct matches where label == 1)
for item in dataset:
    if item["label"] == 1:
        target_word = item["target_word"]
        sense_id = item["sense_id"]
        sentence_counts[target_word][sense_id] += 1

# 5. Print the final summary cleanly
print("\n" + "=" * 40)
print("🎯 SENTENCE COUNT PER SENSE")
print("=" * 40)

for word, senses in sentence_counts.items():
    print(f"\nWord: '{word}'")
    for sense_id, count in senses.items():
        print(f"  ➔ Sense {sense_id}: {count} sentences")

print("\n" + "=" * 40)
print(f"Total rows in the entire dataset: {len(dataset)}")

📤 Please upload 'marathi_wsd_dataset_final.json'


Saving marathi_wsd_dataset_final.json to marathi_wsd_dataset_final.json

🎯 SENTENCE COUNT PER SENSE

Word: 'डोळा'
  ➔ Sense 101: 92 sentences
  ➔ Sense 102: 98 sentences
  ➔ Sense 103: 106 sentences
  ➔ Sense 104: 106 sentences
  ➔ Sense 105: 92 sentences

Word: 'मार्ग'
  ➔ Sense 201: 262 sentences
  ➔ Sense 202: 115 sentences
  ➔ Sense 203: 160 sentences

Word: 'वाट'
  ➔ Sense 301: 430 sentences
  ➔ Sense 302: 115 sentences
  ➔ Sense 303: 246 sentences

Word: 'वाटणे'
  ➔ Sense 401: 123 sentences
  ➔ Sense 402: 100 sentences
  ➔ Sense 403: 100 sentences

Word: 'देणे'
  ➔ Sense 501: 662 sentences
  ➔ Sense 502: 393 sentences
  ➔ Sense 503: 2484 sentences

Word: 'भाव'
  ➔ Sense 601: 114 sentences
  ➔ Sense 602: 207 sentences
  ➔ Sense 603: 136 sentences

Word: 'योग'
  ➔ Sense 701: 252 sentences
  ➔ Sense 702: 115 sentences
  ➔ Sense 703: 169 sentences

Word: 'अर्थ'
  ➔ Sense 801: 132 sentences
  ➔ Sense 802: 294 sentences
  ➔ Sense 803: 115 sentences

Word: 'नाव'
  ➔ Sense 1001: 301 sent

In [ ]:
import json
from google.colab import files

# 1. Load your current dataset
input_file = "marathi_wsd_dataset_final.json"
output_file = "marathi_wsd_dataset_filtered.json"

print("Loading dataset...")
with open(input_file, "r", encoding="utf-8") as f:
    dataset = json.load(f)

print(f"Original dataset size: {len(dataset)} rows")

# 2. Define the words you want to drop
words_to_drop = ["उडणे", "फोडणे", "गंध","हलका"]

# 3. Filter out those words
filtered_dataset = [
    item for item in dataset
    if item["target_word"] not in words_to_drop
]

# 4. Save the new, clean dataset
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(filtered_dataset, f, ensure_ascii=False, indent=2)

print(f"\n✅ Successfully removed: {words_to_drop}")
print(f"New dataset size: {len(filtered_dataset)} rows")

# 5. Download the new file
print(f"Downloading {output_file}...")
files.download(output_file)

Loading dataset...
Original dataset size: 50893 rows

✅ Successfully removed: ['उडणे', 'फोडणे', 'गंध', 'हलका']
New dataset size: 50242 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json
from collections import defaultdict
from google.colab import files

# 1. Prompt you to upload the dataset
print("=" * 50)
print("📤 Please upload 'marathi_wsd_dataset_final.json'")
print("=" * 50)

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2. Load the uploaded JSON file
with open(file_name, "r", encoding="utf-8") as f:
    dataset = json.load(f)

# 3. Create a dictionary to store the counts
sentence_counts = defaultdict(lambda: defaultdict(int))

# 4. Count the sentences (Only counting the correct matches where label == 1)
for item in dataset:
    if item["label"] == 1:
        target_word = item["target_word"]
        sense_id = item["sense_id"]
        sentence_counts[target_word][sense_id] += 1

# 5. Print the final summary cleanly
print("\n" + "=" * 40)
print("🎯 SENTENCE COUNT PER SENSE")
print("=" * 40)

for word, senses in sentence_counts.items():
    print(f"\nWord: '{word}'")
    for sense_id, count in senses.items():
        print(f"  ➔ Sense {sense_id}: {count} sentences")

print("\n" + "=" * 40)
print(f"Total rows in the entire dataset: {len(dataset)}")

📤 Please upload 'marathi_wsd_dataset_final.json'


Saving marathi_wsd_dataset_filtered (1).json to marathi_wsd_dataset_filtered (1) (1).json

🎯 SENTENCE COUNT PER SENSE

Word: 'डोळा'
  ➔ Sense 101: 92 sentences
  ➔ Sense 102: 98 sentences
  ➔ Sense 103: 106 sentences
  ➔ Sense 104: 106 sentences
  ➔ Sense 105: 92 sentences

Word: 'मार्ग'
  ➔ Sense 201: 262 sentences
  ➔ Sense 202: 115 sentences
  ➔ Sense 203: 160 sentences

Word: 'वाट'
  ➔ Sense 301: 430 sentences
  ➔ Sense 302: 115 sentences
  ➔ Sense 303: 246 sentences

Word: 'वाटणे'
  ➔ Sense 401: 123 sentences
  ➔ Sense 402: 100 sentences
  ➔ Sense 403: 100 sentences

Word: 'देणे'
  ➔ Sense 501: 662 sentences
  ➔ Sense 502: 393 sentences
  ➔ Sense 503: 2484 sentences

Word: 'भाव'
  ➔ Sense 601: 114 sentences
  ➔ Sense 602: 207 sentences
  ➔ Sense 603: 136 sentences

Word: 'योग'
  ➔ Sense 701: 252 sentences
  ➔ Sense 702: 115 sentences
  ➔ Sense 703: 169 sentences

Word: 'अर्थ'
  ➔ Sense 801: 132 sentences
  ➔ Sense 802: 294 sentences
  ➔ Sense 803: 115 sentences

Word: 'नाव'
  ➔ Se